# Stage 5 — User Voice Encoding (Colab GPU)

Encodes `user_voice_docs.parquet` → `user_voice_embeddings.npy` using `hyp1231/blair-roberta-large`.

**Runtime:** ~20-30 min on T4 GPU for ~80k tier-2+ users.

**Before running:**
1. Run `python -m src.stage5_users.main` locally to produce `user_voice_docs.parquet`
2. Set Runtime → Change runtime type → GPU (T4)
3. Upload `data/processed/user_voice_docs.parquet` when prompted in Cell 3

In [ ]:
# Cell 1 — Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Switch to GPU runtime before proceeding.')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers torch pandas numpy pyarrow tqdm

In [ ]:
# Cell 3 — Upload user_voice_docs.parquet
from google.colab import files
import pandas as pd

print('Upload user_voice_docs.parquet from data/processed/ on your local machine:')
uploaded = files.upload()

# Load the uploaded file
voice_docs_df = pd.read_parquet('user_voice_docs.parquet')
print(f'Loaded {len(voice_docs_df):,} voice documents')
print(f'Columns: {voice_docs_df.columns.tolist()}')
print(f'Sample doc (first 300 chars):')
print(voice_docs_df['voice_document'].iloc[0][:300])

In [ ]:
# Cell 4 — Encode user voices with BLAIR
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

MODEL_NAME  = 'hyp1231/blair-roberta-large'
BATCH_SIZE  = 32
MAX_SEQ_LEN = 512
CHECKPOINT_EVERY = 5000  # save partial results every N users

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print(f'Loading model: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
model = model.to(device)
print('Model loaded.')

docs     = voice_docs_df['voice_document'].tolist()
user_ids = voice_docs_df['user_id'].tolist()
n        = len(docs)
print(f'Encoding {n:,} user voice documents ...')

emb_list = []
checkpoint_count = 0

with torch.no_grad():
    for start in tqdm(range(0, n, BATCH_SIZE), desc='Encoding', unit='batch'):
        batch_docs = docs[start: start + BATCH_SIZE]
        enc = tokenizer(
            batch_docs,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        # CLS token embedding (same as BLAIR item encoding)
        cls_emb = out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)
        # L2 normalize
        norms = np.linalg.norm(cls_emb, axis=1, keepdims=True)
        norms = np.where(norms < 1e-9, 1.0, norms)
        cls_emb = cls_emb / norms
        emb_list.append(cls_emb)

        # Checkpoint every CHECKPOINT_EVERY users
        encoded_so_far = start + len(batch_docs)
        if encoded_so_far // CHECKPOINT_EVERY > checkpoint_count:
            checkpoint_count = encoded_so_far // CHECKPOINT_EVERY
            partial_embs = np.concatenate(emb_list, axis=0)
            np.save('user_voice_embeddings_partial.npy', partial_embs)
            np.save('user_voice_ids_partial.npy', np.array(user_ids[:len(partial_embs)], dtype=object))
            print(f'  Checkpoint saved: {len(partial_embs):,} users encoded')

embeddings = np.concatenate(emb_list, axis=0)
print(f'Encoding complete: shape={embeddings.shape}')
print(f'Norm check: min={np.linalg.norm(embeddings, axis=1).min():.4f}  max={np.linalg.norm(embeddings, axis=1).max():.4f}  (should be ~1.0)')

# Save final files
np.save('user_voice_embeddings.npy', embeddings)
np.save('user_voice_ids.npy', np.array(user_ids, dtype=object))
print('Saved user_voice_embeddings.npy and user_voice_ids.npy')

In [ ]:
# Cell 5 — Verify
import numpy as np

embs = np.load('user_voice_embeddings.npy')
ids  = np.load('user_voice_ids.npy', allow_pickle=True)

print('=== Verification ===')
print(f'Embeddings shape : {embs.shape}   (expected: (N, 1024))')
print(f'IDs shape        : {ids.shape}')
print(f'Dtype            : {embs.dtype}  (expected: float32)')

norms = np.linalg.norm(embs, axis=1)
print(f'L2 norms: min={norms.min():.6f}  max={norms.max():.6f}  mean={norms.mean():.6f}  (all should be ~1.0)')

# Sample cosine similarities between 5 random pairs
print()
print('Sample cosine similarities (random pairs):')
rng = np.random.default_rng(42)
idx = rng.choice(len(embs), size=(5, 2), replace=False)
for i, j in idx:
    sim = float(np.dot(embs[i], embs[j]))
    print(f'  user {ids[i][:12]}... vs {ids[j][:12]}...: sim={sim:.4f}')

print()
print('Verification passed. Ready to download.')

In [ ]:
# Cell 6 — Download outputs
from google.colab import files

print('Downloading user_voice_embeddings.npy ...')
files.download('user_voice_embeddings.npy')

print('Downloading user_voice_ids.npy ...')
files.download('user_voice_ids.npy')

print()
print('Place both files in data/embeddings/ on your local machine before running Stage 6.')